Fine-tune the AHL-D CRNN on Fidel handwritten (writer-level split), then test on Fidel test and AHL-D test.

In [ ]:
#@title Fine-Tune AHL-D 29K CRNN on Fidel Handwritten and Test on Fidel + AHL-D

import os
import zipfile
import copy
import random
import json
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from jiwer import cer, wer


CHECKPOINT_PATH = (
    "/content/drive/MyDrive/"
    "amharic_htr_research/ocr/research/crnn/"
    "ahld29k/best_checkpoint.pt"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "amharic_htr_research/ocr/research/crnn/"
    "ahld29k_fidel_finetune"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

FIDEL_ROOT = "/content/fidel_dataset"
FIDEL_ZIP = os.path.join(FIDEL_ROOT, "train-clean.zip")
FIDEL_LABELS = os.path.join(FIDEL_ROOT, "train_labels.csv")
FIDEL_IMAGES_ROOT = os.path.join(FIDEL_ROOT, "train-clean")

FIDEL_BASE_URL = (
    "https://huggingface.co/datasets/upanzi/fidel-dataset/"
    "resolve/main/"
)

FINETUNE_EPOCHS = 10
FINETUNE_BATCH_SIZE = 16
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-5
PATIENCE = 3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

state_dict = checkpoint["model_state_dict"]
char_to_idx = checkpoint["char_to_idx"]

idx_to_char = {
    int(k): v
    for k, v in checkpoint["idx_to_char"].items()
}

blank_idx = checkpoint["blank_idx"]


class CRNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.MaxPool2d((2, 1), (2, 1)),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.MaxPool2d((2, 1), (2, 1))
        )

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.2
        )

        self.classifier = nn.Linear(
            512,
            num_classes
        )

    def forward(self, x):

        x = self.cnn(x)

        x = F.adaptive_avg_pool2d(
            x,
            (1, x.size(3))
        )

        x = x.squeeze(2)

        x = x.permute(2, 0, 1)

        x, _ = self.rnn(x)

        return self.classifier(x)


model = CRNN(
    num_classes=len(char_to_idx) + 1
).to(DEVICE)

model.load_state_dict(state_dict)
model.eval()


print(
    f"Model loaded | "
    f"epoch {checkpoint.get('epoch')} | "
    f"AHL-D CER {checkpoint.get('best_cer', 0):.4f}"
)


if not os.path.exists(FIDEL_LABELS):

    !wget -q --show-progress \
        -O "{FIDEL_LABELS}" \
        "{FIDEL_BASE_URL}train_labels.csv"


if not os.path.exists(FIDEL_ZIP):

    !wget -q --show-progress \
        -O "{FIDEL_ZIP}" \
        "{FIDEL_BASE_URL}train-clean.zip"


labels = pd.read_csv(
    FIDEL_LABELS
)

labels = labels[
    labels["type"].astype(str).str.lower()
    == "handwritten"
].reset_index(drop=True)


if len(labels) == 0:
    raise RuntimeError(
        "No handwritten Fidel samples found."
    )


if not os.path.exists(FIDEL_IMAGES_ROOT):

    with zipfile.ZipFile(
        FIDEL_ZIP,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            FIDEL_ROOT
        )


image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff"
}

image_lookup = {}

for root, dirs, files in os.walk(FIDEL_ROOT):

    for filename in files:

        extension = os.path.splitext(
            filename
        )[1].lower()

        if extension not in image_extensions:
            continue

        path = os.path.join(
            root,
            filename
        )

        image_lookup[filename] = path

        image_lookup[
            os.path.splitext(filename)[0]
        ] = path


matched_rows = []

for _, row in labels.iterrows():

    identifier = str(
        row["image_filename"]
    )

    candidates = [
        identifier,
        os.path.basename(identifier),
        os.path.splitext(
            os.path.basename(identifier)
        )[0]
    ]

    image_path = None

    for candidate in candidates:

        if candidate in image_lookup:

            image_path = image_lookup[
                candidate
            ]

            break

    if image_path is not None:

        matched_rows.append({
            "image_path": image_path,
            "text": str(row["line_text"]),
            "writer": str(row["writer"])
        })


fidel_df = pd.DataFrame(
    matched_rows
)


writers = sorted(
    fidel_df["writer"].unique()
)

random.shuffle(writers)

n_writers = len(writers)

train_writer_count = int(
    n_writers * 0.80
)

val_writer_count = int(
    n_writers * 0.10
)

train_writers = set(
    writers[:train_writer_count]
)

val_writers = set(
    writers[
        train_writer_count:
        train_writer_count + val_writer_count
    ]
)

test_writers = set(
    writers[
        train_writer_count + val_writer_count:
    ]
)


fidel_train_df = fidel_df[
    fidel_df["writer"].isin(train_writers)
].reset_index(drop=True)

fidel_val_df = fidel_df[
    fidel_df["writer"].isin(val_writers)
].reset_index(drop=True)

fidel_test_df = fidel_df[
    fidel_df["writer"].isin(test_writers)
].reset_index(drop=True)


print(
    f"Fidel train: {len(fidel_train_df)} | "
    f"val: {len(fidel_val_df)} | "
    f"test: {len(fidel_test_df)}"
)


class FidelFineTuneDataset(Dataset):

    def __init__(self, dataframe):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("L")

        image = image.resize(
            (
                IMG_WIDTH,
                IMG_HEIGHT
            ),
            Image.Resampling.LANCZOS
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        text = str(row["text"])

        target = torch.tensor(
            [
                char_to_idx[c]
                for c in text
                if c in char_to_idx
            ],
            dtype=torch.long
        )

        return image, target, text


def fidel_collate_fn(batch):

    images = torch.stack(
        [item[0] for item in batch]
    )

    targets = torch.cat(
        [item[1] for item in batch]
    )

    target_lengths = torch.tensor(
        [
            len(item[1])
            for item in batch
        ],
        dtype=torch.long
    )

    texts = [
        item[2]
        for item in batch
    ]

    return (
        images,
        targets,
        target_lengths,
        texts
    )


fidel_train_loader = DataLoader(
    FidelFineTuneDataset(fidel_train_df),
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=fidel_collate_fn
)

fidel_val_loader = DataLoader(
    FidelFineTuneDataset(fidel_val_df),
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=fidel_collate_fn
)

fidel_test_loader = DataLoader(
    FidelFineTuneDataset(fidel_test_df),
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=fidel_collate_fn
)


def fidel_greedy_decode(outputs):

    predictions = outputs.argmax(
        dim=2
    )

    results = []

    for sequence in predictions.permute(1, 0):

        text = []
        previous = blank_idx

        for index in sequence.tolist():

            if (
                index != blank_idx
                and index != previous
            ):

                text.append(
                    idx_to_char.get(
                        index,
                        ""
                    )
                )

            previous = index

        results.append(
            "".join(text)
        )

    return results


def evaluate_fidel(
    model,
    loader
):

    model.eval()

    references = []
    predictions = []

    with torch.no_grad():

        for (
            images,
            targets,
            target_lengths,
            texts
        ) in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            outputs = model(images)

            predictions.extend(
                fidel_greedy_decode(outputs)
            )

            references.extend(texts)

    return {
        "cer": float(
            cer(
                references,
                predictions
            )
        ),
        "wer": float(
            wer(
                references,
                predictions
            )
        ),
        "exact_accuracy": float(
            sum(
                r == p
                for r, p in zip(
                    references,
                    predictions
                )
            ) / len(references)
        ),
        "references": references,
        "predictions": predictions
    }


criterion = nn.CTCLoss(
    blank=blank_idx,
    zero_infinity=True
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


best_val_cer = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0


for epoch in range(
    1,
    FINETUNE_EPOCHS + 1
):

    model.train()

    epoch_loss = 0.0
    batch_count = 0

    for (
        images,
        targets,
        target_lengths,
        texts
    ) in fidel_train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )

        target_lengths = target_lengths.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(images)

        input_lengths = torch.full(
            size=(images.size(0),),
            fill_value=outputs.size(0),
            dtype=torch.long,
            device=DEVICE
        )

        log_probs = F.log_softmax(
            outputs,
            dim=2
        )

        loss = criterion(
            log_probs,
            targets,
            input_lengths,
            target_lengths
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1

    train_loss = (
        epoch_loss / batch_count
    )

    val_metrics = evaluate_fidel(
        model,
        fidel_val_loader
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {train_loss:.4f} | "
        f"Fidel Val CER {val_metrics['cer']:.4f} | "
        f"Fidel Val WER {val_metrics['wer']:.4f}"
    )

    if val_metrics["cer"] < best_val_cer:

        best_val_cer = val_metrics["cer"]
        best_epoch = epoch

        best_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": best_state,
                "epoch": epoch,
                "best_fidel_val_cer": best_val_cer,
                "char_to_idx": char_to_idx,
                "idx_to_char": {
                    str(k): v
                    for k, v in idx_to_char.items()
                },
                "blank_idx": blank_idx,
                "source_checkpoint": CHECKPOINT_PATH,
                "training_dataset": "Fidel handwritten"
            },
            os.path.join(
                OUTPUT_DIR,
                "best_fidel_finetuned_checkpoint.pt"
            )
        )

    else:

        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:

        print(
            f"Early stopping at epoch {epoch}"
        )

        break


model.load_state_dict(
    best_state
)

model.eval()


fidel_test_metrics = evaluate_fidel(
    model,
    fidel_test_loader
)


fidel_predictions_path = os.path.join(
    OUTPUT_DIR,
    "fidel_test_predictions.csv"
)

pd.DataFrame({
    "ground_truth":
        fidel_test_metrics["references"],
    "prediction":
        fidel_test_metrics["predictions"]
}).to_csv(
    fidel_predictions_path,
    index=False,
    encoding="utf-8-sig"
)


ahld_metrics = evaluate_model(
    model,
    test_loader
)


ahld_predictions_path = os.path.join(
    OUTPUT_DIR,
    "ahld_test_predictions_after_fidel_finetune.csv"
)

save_predictions(
    ahld_metrics["references"],
    ahld_metrics["predictions"],
    ahld_predictions_path
)


final_metrics = {

    "best_epoch": best_epoch,

    "best_fidel_validation_cer":
        best_val_cer,

    "fidel_test": {
        "samples":
            len(
                fidel_test_metrics["references"]
            ),
        "cer":
            fidel_test_metrics["cer"],
        "wer":
            fidel_test_metrics["wer"],
        "exact_accuracy":
            fidel_test_metrics["exact_accuracy"]
    },

    "ahld_test": {
        "samples":
            len(
                ahld_metrics["references"]
            ),
        "cer":
            ahld_metrics["cer"],
        "wer":
            ahld_metrics["wer"],
        "exact_accuracy":
            ahld_metrics["exact_accuracy"]
    }
}


with open(
    os.path.join(
        OUTPUT_DIR,
        "finetune_test_metrics.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )


print()
print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Best Fidel val CER: "
    f"{best_val_cer:.4f}"
)

print(
    f"Fidel test | "
    f"CER {fidel_test_metrics['cer']:.4f} | "
    f"WER {fidel_test_metrics['wer']:.4f}"
)

print(
    f"AHL-D test | "
    f"CER {ahld_metrics['cer']:.4f} | "
    f"WER {ahld_metrics['wer']:.4f}"
)

print(
    f"Checkpoint: "
    f"{os.path.join(OUTPUT_DIR, 'best_fidel_finetuned_checkpoint.pt')}"
)

print(
    f"Fidel predictions: "
    f"{fidel_predictions_path}"
)

print(
    f"AHL-D predictions: "
    f"{ahld_predictions_path}"
)

Model loaded | epoch 20 | AHL-D CER 0.0462
Fidel train: 27855 | val: 3458 | test: 3507
Epoch 01 | Loss 2.9406 | Fidel Val CER 0.3934 | Fidel Val WER 0.8395
Epoch 02 | Loss 2.0716 | Fidel Val CER 0.3479 | Fidel Val WER 0.7907
Epoch 03 | Loss 1.8019 | Fidel Val CER 0.3207 | Fidel Val WER 0.7547
Epoch 04 | Loss 1.6513 | Fidel Val CER 0.3021 | Fidel Val WER 0.7265
Epoch 05 | Loss 1.5553 | Fidel Val CER 0.2869 | Fidel Val WER 0.6980
Epoch 06 | Loss 1.4879 | Fidel Val CER 0.2737 | Fidel Val WER 0.6726
Epoch 07 | Loss 1.4375 | Fidel Val CER 0.2667 | Fidel Val WER 0.6590
Epoch 08 | Loss 1.3967 | Fidel Val CER 0.2590 | Fidel Val WER 0.6432
Epoch 09 | Loss 1.3637 | Fidel Val CER 0.2529 | Fidel Val WER 0.6308
Epoch 10 | Loss 1.3354 | Fidel Val CER 0.2472 | Fidel Val WER 0.6189

Best epoch: 10
Best Fidel val CER: 0.2472
Fidel test | CER 0.2797 | WER 0.6698
AHL-D test | CER 0.2369 | WER 0.4457
Checkpoint: /content/drive/MyDrive/amharic_htr_research/ocr/research/crnn/ahld29k_fidel_finetune/best_fide